In [23]:
"""
    invertir_bits(k, num_bits)

Invierte el orden de los bits de un entero `k` considerando un total de `num_bits`.
"""
function invertir_bits(k::Int, num_bits::Int)
    resultado = 0
    for i in 0:(num_bits - 1)
        if ((k >> i) & 1) == 1
            resultado |= (1 << (num_bits - 1 - i))
        end
    end
    return resultado
end

invertir_bits

In [24]:
"""
    calcular_K2(K, q, p)
K2 = k_q * 2^p + ... + k_p * 2^q
Es decir, toma los bits de K desde la posición q hasta p, y los invierte en ese rango.
"""
function calcular_K2(K::Int, q::Int, p::Int)
    K2 = 0
    #Iteramos i desde q hasta p
    for i in q:p
        #Extraxion el bit i-ésimo de K
        bit_val = (K >> i) & 1
        #Calculamos la posición destino en K2
        pos_destino = p - (i - q)
        
        if bit_val == 1
            K2 |= (1 << pos_destino)
        end
    end
    return K2
end

calcular_K2

In [25]:
function Transformada_Rapida_Fourier(y::Vector)
    N = length(y)
    total_bits = Int(log2(N))
    if 2^total_bits != N
        error("La longitud N debe ser potencia de 2.")
    end
    
    m = div(N, 2)
    p = total_bits - 1
    
    # Paso 1
    M = m
    q = p
    zeta = exp(π * im / m)
    
    # Paso 2
    c = Complex{Float64}.(y)
    
    # Paso 3, potencias de zeta
    xi = [zeta^j for j in 0:(2*m - 1)]
    
    #Bucles principales
    K = 0
    
    for L in 1:(p + 1)
        while K < 2*m - 1
            for j in 1:M
                # k_idx, indice "K" actual
                k_idx = K + j - 1
                
                #Calcular K2 usando la descomposición de bits
                K2 = calcular_K2(k_idx, q, p)
                
                #Operación Mariposa
                idx_c_K = k_idx + 1
                idx_c_KM = k_idx + M + 1
                idx_xi = K2 + 1
                
                eta = c[idx_c_KM] * xi[idx_xi]
                
                c[idx_c_KM] = c[idx_c_K] - eta
                c[idx_c_K] = c[idx_c_K] + eta
            end
            
            K = K + 2*M
        end
        
        #Reinicio para siguiente nivel L
        K = 0
        M = div(M, 2)
        q = q - 1
    end
    
    #Reordenamiento
    for K_idx in 0:(2*m - 1)
        j_idx = invertir_bits(K_idx, total_bits)
        if j_idx > K_idx
            c[j_idx + 1], c[K_idx + 1] = c[K_idx + 1], c[j_idx + 1]
        end
    end
    
    #Coeficientes Trigonométricos Finales
    a = zeros(Float64, m + 1)
    b = zeros(Float64, m)
    
    # a0
    a[1] = real(c[1]) / m
    
    # am = Re(e^(-i*pi*m) * cm / m) -> (-1)^m * real(cm)/m
    signo = (m % 2 == 0) ? 1.0 : -1.0
    a[m+1] = real(c[m+1]) / m * signo
    
    # Resto aj, bj
    for j in 1:(m - 1)
        factor = exp(-im * π * j) * c[j+1] / m
        a[j+1] = real(factor)
        b[j]   = imag(factor)
    end
    
    return c, a, b
end

Transformada_Rapida_Fourier (generic function with 1 method)

In [26]:
using Printf

#Funcion de ejemplo
f_ejemplo(x) = 2*x^2 - 9
N = 4
m = 2
x_nodos = [-π + j*(π/m) for j in 0:(N-1)]
y_datos = f_ejemplo.(x_nodos)

println("Entrada")
for i in 1:N
    @printf("j=%d, x=%.4f, f(x)=%.4f\n", i-1, x_nodos[i], y_datos[i])
end

# Transformada
c_res, a_res, b_res = Transformada_Rapida_Fourier(y_datos)

println("\nResultados")
println("a_0 = $(a_res[1])")
println("a_1 = $(a_res[2])")
println("a_2 = $(a_res[3])")
println("b_1 = $(b_res[1])")

Entrada
j=0, x=-3.1416, f(x)=10.7392
j=1, x=-1.5708, f(x)=-4.0652
j=2, x=0.0000, f(x)=-9.0000
j=3, x=1.5708, f(x)=-4.0652

Resultados
a_0 = -3.195593398365963
a_1 = -9.869604401089358
a_2 = 4.934802200544679
b_1 = -1.2086779438644711e-15
